In [111]:
import json 

with open("aiod_model.json", "r") as f:
    conceptual_model = json.load(f)

In [112]:
for i in range(len(conceptual_model['classes'])):
    if conceptual_model['classes'][i]['name'] == 'Event':
        print(i)

54


In [27]:
conceptual_model['classes'][54]["inherited_properties"][0]
conceptual_model['classes'][54]["direct_properties"][0]

{'name': 'organiser',
 'domain': ['Event'],
 'range': ['Agent'],
 'annotations': {'comment': 'The entity organising the Event.',
  'label': 'organiser'},
 'equivalent_properties': []}

Generate schema

In [4]:
# Jupyter only looks in its default paths (sys.path does not automatically include sibling directories).
import sys
import os

# Define the absolute path to `aiod/src`
SRC_PATH = os.path.abspath("../src")

# Add it to Python's module search path if it's not already there
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f"✅ Added {SRC_PATH} to sys.path")

✅ Added /home/taniya_das/Documents/AIOD-rest-api/src to sys.path


In [5]:
# Example schema generation
from database.model.ai_asset.ai_asset import AIAssetBase

json.loads(AIAssetBase.schema_json())
AIAssetBase.schema_json()

/home/taniya_das/Documents/AIOD-rest-api/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/taniya_das/Documents/AIOD-rest-api/venv/lib/python3.11/site-packages/sqlmodel/main.py:638: SAWarning: This declarative base already contains a class with the same class name and module name as database.model.helper_functions.LinkTable, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)


'{"title": "AIAssetBase", "type": "object", "properties": {"platform": {"title": "Platform", "description": "The external platform from which this resource originates. Leave empty if this item originates from AIoD. If platform is not None, the platform_resource_identifier should be set as well.", "maxLength": 64, "example": "example", "type": "string"}, "platform_resource_identifier": {"title": "Platform Resource Identifier", "description": "A unique identifier issued by the external platform that\'s specified in \'platform\'. Leave empty if this item is not part of an external platform. For example, for HuggingFace, this should be the <namespace>/<dataset_name>, and for Openml, the OpenML identifier.", "maxLength": 256, "example": "1", "type": "string"}, "name": {"title": "Name", "maxLength": 256, "example": "The name of this resource", "type": "string"}, "date_published": {"title": "Date Published", "description": "The datetime (utc) on which this resource was first published on an e

In [6]:
# generate complete schema 

import os
import json
import importlib.util
from pathlib import Path
from pydantic import BaseModel
from sqlmodel import SQLModel

# base directory 
BASE_DIR = "database/model"
OUTPUT_FILE = "schemas.json"

# dictionary to hold all schemas
all_schemas = {}
read_error = {}
class_parents = {}

for root, _, files in os.walk(os.path.join(SRC_PATH, BASE_DIR)):
    for file in files:
        if file.endswith(".py") and not file.startswith("__"):
            
            module_path = os.path.join(root, file)
            module_name = Path(module_path).with_suffix("").as_posix().split("/")[-1]
            
            try:
               
                spec = importlib.util.spec_from_file_location(module_name, module_path)
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)

                
                for attr_name in dir(module):
                    
                    SQLModel.metadata.clear()  
                    
                    attr = getattr(module, attr_name)
                    # Only process Pydantic models (subclasses of BaseModel)
                    if isinstance(attr, type) and issubclass(attr, BaseModel) and attr is not BaseModel:
                        
                        # if attr.__name__.endswith("Base") or attr.__name__.endswith("Table") or attr.__name__.endswith("ORM"):  # Exclude abstract Base classes
                        #     continue
                     
                        schema = attr.schema_json()
                        all_schemas[attr.__name__] = json.loads(schema)
                        
                        # Get parent classes dynamically
                        parent_classes = [
                            parent.__name__ for parent in attr.__bases__
                            if isinstance(parent, type) and issubclass(parent, BaseModel) and parent is not BaseModel
                        ]
                        
                        # Store parent mapping
                        class_parents[attr.__name__] = parent_classes

            except Exception as e:
                print(f"Error processing {module_name}: {e}")
                read_error.update({module_name:e})


# with open(OUTPUT_FILE, "w") as f:
#     json.dump(all_schemas, f, indent=4)

# print(f"All schemas have been written to {OUTPUT_FILE}")


/home/taniya_das/Documents/AIOD-rest-api/venv/lib/python3.11/site-packages/sqlmodel/main.py:638: SAWarning: This declarative base already contains a class with the same class name and module name as database.model.ai_asset.distribution.DistributionORM, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)
/home/taniya_das/Documents/AIOD-rest-api/venv/lib/python3.11/site-packages/sqlmodel/main.py:638: SAWarning: This declarative base already contains a class with the same class name and module name as database.model.ai_resource.note.NoteORM, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)
/home/taniya_das/Documents/AIOD-rest-api/venv/lib/python3.11/site-packages/sqlmodel/main.py:638: SAWarning: This declarative base already contains a class with the same class name and module name as database.model.models_and_experiments.runnable_distribution.RunnableDistributionORM,

In [7]:
read_error

{}

In [8]:
class_parents

{'NamedRelation': ['SQLModel'],
 'SQLModel': [],
 'AIoDConceptBase': ['SQLModel'],
 'Distribution': ['DistributionBase'],
 'DistributionBase': ['AIoDConceptBase'],
 'AIAssetTable': ['SQLModel'],
 'License': ['NamedRelation'],
 'AIAsset': ['AIAssetBase', 'AbstractAIResource'],
 'AIAssetBase': ['AIResourceBase'],
 'AIResourceBase': ['AIoDConceptBase'],
 'AbstractAIResource': ['AIResourceBase', 'AIoDConcept'],
 'Address': ['AddressBase'],
 'AddressBase': ['SQLModel'],
 'AddressORM': ['AddressBase'],
 'Geo': ['GeoBase'],
 'GeoBase': ['SQLModel'],
 'GeoORM': ['GeoBase'],
 'Location': ['LocationBase'],
 'LocationBase': ['SQLModel'],
 'LocationORM': ['LocationBase'],
 'Language': ['NamedRelation'],
 'AgentTable': ['SQLModel'],
 'OrganisationType': ['NamedRelation'],
 'Agent': ['AgentBase', 'AbstractAIResource'],
 'AgentBase': ['AIResourceBase'],
 'Email': ['NamedRelation'],
 'Contact': ['ContactBase', 'AIoDConcept'],
 'Organisation': ['OrganisationBase', 'Agent'],
 'OrganisationBase': ['Agent

In [21]:
all_inheritance = {key: set(value) for key, value in class_parents.items()}  # sets to avoid duplicates

# Keep iterating until all inherited classes are fully expanded
for key in class_parents:
    queue = list(class_parents[key])  # Start with direct parents

    while queue:
        parent = queue.pop(0)
        if parent in class_parents:
            new_parents = class_parents[parent]  # Get parent's parents
            queue.extend(new_parents)  # Add them to queue for further processing
            all_inheritance[key].update(new_parents)  # Add to final set

# Convert sets back to lists for JSON compatibility
all_inheritance = {k: list(v) for k, v in all_inheritance.items()}

all_inheritance


{'NamedRelation': ['SQLModel'],
 'SQLModel': [],
 'AIoDConceptBase': ['SQLModel'],
 'Distribution': ['DistributionBase', 'SQLModel', 'AIoDConceptBase'],
 'DistributionBase': ['SQLModel', 'AIoDConceptBase'],
 'AIAssetTable': ['SQLModel'],
 'License': ['NamedRelation', 'SQLModel'],
 'AIAsset': ['SQLModel',
  'AIResourceBase',
  'AIoDConcept',
  'AbstractAIResource',
  'AIAssetBase',
  'AIoDConceptBase'],
 'AIAssetBase': ['AIResourceBase', 'SQLModel', 'AIoDConceptBase'],
 'AIResourceBase': ['SQLModel', 'AIoDConceptBase'],
 'AbstractAIResource': ['AIResourceBase',
  'SQLModel',
  'AIoDConcept',
  'AIoDConceptBase'],
 'Address': ['SQLModel', 'AddressBase'],
 'AddressBase': ['SQLModel'],
 'AddressORM': ['SQLModel', 'AddressBase'],
 'Geo': ['SQLModel', 'GeoBase'],
 'GeoBase': ['SQLModel'],
 'GeoORM': ['SQLModel', 'GeoBase'],
 'Location': ['SQLModel', 'LocationBase'],
 'LocationBase': ['SQLModel'],
 'LocationORM': ['SQLModel', 'LocationBase'],
 'Language': ['NamedRelation', 'SQLModel'],
 'Agen

In [9]:
all_schemas["AIAsset"].keys()

dict_keys(['title', 'type', 'properties', 'required'])

In [24]:
import re

generated_schema = all_schemas.copy()

def preprocess_generated_schema(generated_schema):
    """
    Preprocess the generated schema to:
    - Merge base classes (ending in Base, Table, ORM) into main classes
    - Keep the original class name (e.g., AIAsset stays AIAsset)
    - Copy properties from base classes to main classes without duplication
    - Rename "title" to "name"
    - Add an empty "equivalent_classes" field
    - Convert "properties" into a dictionary format
    - Ensure entity names are case-consistent
    """
    preprocessed_schema = {}
    merged_classes = {}  # Stores final merged entities
    original_case_map = {}  # To track original casing of entity names

    # Step 1: Identify mappings of base-like classes to main classes
    class_mappings = {}  # { "aiassetbase": "aiasset", "aiassettable": "aiasset", "aiassetorm": "aiasset" }

    for entity_name in generated_schema:
        normalized_name = entity_name.lower()
        original_case_map[normalized_name] = entity_name  # Store original case

        match = re.match(r"(.+)(Base|Table|ORM)$", entity_name, re.IGNORECASE)
        if match:
            main_class_name = match.group(1)  # Extract base name without suffix
            class_mappings[normalized_name] = main_class_name.lower()  # Store mapping

    # Step 2: Merge properties into the main class
    for entity_name, schema in generated_schema.items():
        normalized_name = entity_name.lower()

        # Determine the final entity name (handling merged classes)
        main_entity_name = class_mappings.get(normalized_name, normalized_name)

        # Use the original case for entity names
        final_entity_name = original_case_map.get(main_entity_name, main_entity_name)

        # Initialize main entity if not already added
        if final_entity_name not in merged_classes:
            merged_classes[final_entity_name] = {
                "name": final_entity_name,  # Keep original case
                "equivalent_classes": [],
                "properties": {},
                "required": schema.get("required", [])
            }

        # Copy properties, avoiding duplicates
        for prop_name, prop_details in schema.get("properties", {}).items():
            processed_property = prop_details.copy()
            processed_property["name"] = processed_property.pop("title", prop_name)  # Rename "title" to "name"

            prop_name_cleaned = processed_property["name"].lower()

            # Add property if it doesn't exist
            if prop_name_cleaned not in merged_classes[final_entity_name]["properties"]:
                merged_classes[final_entity_name]["properties"][prop_name_cleaned] = processed_property

    return merged_classes  # Return as dictionary


# Apply preprocessing
preprocessed_generated_schema = preprocess_generated_schema(generated_schema)


KeyError: 'direct_properties'

In [140]:
import re

generated_schema = all_schemas.copy()

def preprocess_generated_schema(generated_schema, all_inheritance):
    """
    Preprocess the generated schema to:
    - Merge base classes (ending in Base, Table, ORM) into main classes
    - Keep the original class name (e.g., AIAsset stays AIAsset)
    - Copy properties from base classes to main classes without duplication
    - Rename "title" to "name"
    - Add "inherited_properties" using all_inheritance
    - Convert "properties" into "direct_properties" dictionary format
    - Ensure entity names are case-consistent
    """
    preprocessed_schema = {}
    merged_classes = {}  # Stores final merged entities
    original_case_map = {}  # To track original casing of entity names

    # Step 1: Identify mappings of base-like classes to main classes
    class_mappings = {}  # { "aiassetbase": "aiasset", "aiassettable": "aiasset", "aiassetorm": "aiasset" }

    for entity_name in generated_schema:
        normalized_name = entity_name.lower()
        original_case_map[normalized_name] = entity_name  # Store original case

        match = re.match(r"(.+)(Base|Table|ORM|Type|Link)$", entity_name, re.IGNORECASE)
        if match:
            main_class_name = match.group(1)  # Extract base name without suffix
            class_mappings[normalized_name] = main_class_name.lower()  # Store mapping

    # Step 2: Merge properties into the main class
    for entity_name, schema in generated_schema.items():
        normalized_name = entity_name.lower()

        # Determine the final entity name (handling merged classes)
        main_entity_name = class_mappings.get(normalized_name, normalized_name)

        # Use the original case for entity names
        final_entity_name = original_case_map.get(main_entity_name, main_entity_name)

        # Initialize main entity if not already added
        if final_entity_name not in merged_classes:
            merged_classes[final_entity_name] = {
                "name": final_entity_name,  # Keep original case
                "equivalent_classes": [],
                "direct_properties": {},  # Renamed from "properties"
                "inherited_properties": {},  # New field for inherited fields
                "required": schema.get("required", [])
            }

        # Copy direct properties, avoiding duplicates
        for prop_name, prop_details in schema.get("properties", {}).items():
            processed_property = prop_details.copy()
            processed_property["name"] = processed_property.pop("title", prop_name)  # Rename "title" to "name"

            prop_name_cleaned = processed_property["name"].lower()

            # Add property to direct_properties if it doesn't exist
            if prop_name_cleaned not in merged_classes[final_entity_name]["direct_properties"]:
                merged_classes[final_entity_name]["direct_properties"][prop_name_cleaned] = processed_property

        # Step 3: Add inherited properties
        inherited_properties = {}
        
        # Ensure the class exists in all_inheritance
        if entity_name in all_inheritance:
            # Iterate over all parent classes for the given entity
            for parent_class in all_inheritance[entity_name]:
                if parent_class in generated_schema:  # Check if parent class exists in the schema
                    parent_schema = generated_schema[parent_class]

                    # Ensure the parent class has properties
                    if "properties" in parent_schema:
                        for prop_name, prop_details in parent_schema["properties"].items():
                            # Ensure the property has a title
                            prop_name_cleaned = prop_details.get("title", prop_name).lower()
                            
                            # Add property only if it does not exist in direct_properties
                            if prop_name_cleaned not in merged_classes[final_entity_name]["direct_properties"]:
                                inherited_properties[prop_name_cleaned] = prop_details.copy()
                                inherited_properties[prop_name_cleaned]["name"] = prop_details.get("title", prop_name)
            
        merged_classes[final_entity_name]["inherited_properties"] = inherited_properties  # Add inherited properties

    return merged_classes  # Return as dictionary


# Apply preprocessing with inheritance
preprocessed_generated_schema = preprocess_generated_schema(generated_schema, all_inheritance)


In [142]:
preprocessed_generated_schema["KnowledgeAsset"]["direct_properties"]

{'platform': {'description': 'The external platform from which this resource originates. Leave empty if this item originates from AIoD. If platform is not None, the platform_resource_identifier should be set as well.',
  'maxLength': 64,
  'example': 'example',
  'type': 'string',
  'name': 'Platform'},
 'platform resource identifier': {'description': "A unique identifier issued by the external platform that's specified in 'platform'. Leave empty if this item is not part of an external platform. For example, for HuggingFace, this should be the <namespace>/<dataset_name>, and for Openml, the OpenML identifier.",
  'maxLength': 256,
  'example': '1',
  'type': 'string',
  'name': 'Platform Resource Identifier'},
 'identifier': {'type': 'integer', 'name': 'Identifier'},
 'date deleted': {'type': 'string',
  'format': 'date-time',
  'name': 'Date Deleted'},
 'aiod entry identifier': {'type': 'integer', 'name': 'Aiod Entry Identifier'},
 'name': {'maxLength': 256,
  'example': 'The name of 

In [129]:
def compare_schemas(conceptual_model, generated_schema):
    """
    Compare conceptual and generated schemas:
    - Identify missing and extra entities
    - Identify missing and extra properties for matching entities
    """
    # Convert both to case-insensitive dictionaries for easy lookup
    conceptual_dict = {entity["name"].lower(): entity for entity in conceptual_model}
    # generated_dict = {entity["name"].lower(): entity for entity in generated_schema}
    generated_dict = {key.lower(): value for key, value in generated_schema.items()}


    # Entities present in both schemas
    matching_entities = set(conceptual_dict.keys()) & set(generated_dict.keys())

    # Entities missing in generated schema
    missing_entities = set(conceptual_dict.keys()) - set(generated_dict.keys())

    # Entities extra in generated schema (not in conceptual model)
    extra_entities = set(generated_dict.keys()) - set(conceptual_dict.keys())

    # Compare properties for matching entities
    missing_properties = {}
    extra_properties = {}

    for entity_name in matching_entities:
        # Get conceptual model properties (merge direct & inherited properties)
        conceptual_properties = set(
            prop["name"].lower() for prop in conceptual_dict[entity_name]["direct_properties"]
        ) | set(
            prop["name"].lower() for prop in conceptual_dict[entity_name]["inherited_properties"]
        )

        # # Get generated schema properties
        # generated_properties = set(
        #     prop["name"].lower() for prop in generated_dict[entity_name]["direct_properties"]
        # )
        
        # Get generated schema properties
        generated_properties = set()

        if entity_name in generated_dict and "direct_properties" in generated_dict[entity_name]:
            generated_properties = {
                generated_dict[entity_name]["direct_properties"][prop]["name"].lower()
                for prop in generated_dict[entity_name]["direct_properties"]
            }

        # Identify missing and extra properties
        missing_props = conceptual_properties - generated_properties
        extra_props = generated_properties - conceptual_properties

        if missing_props:
            missing_properties[entity_name] = list(missing_props)

        if extra_props:
            extra_properties[entity_name] = list(extra_props)

    # Store results
    comparison_results = {
        "matching_entities": list(matching_entities),
        "missing_entities": list(missing_entities),
        "extra_entities": list(extra_entities),
        "missing_properties": missing_properties,
        "extra_properties": extra_properties,
    }

    return comparison_results


# Run comparison
comparison_results = compare_schemas(conceptual_model["classes"], preprocessed_generated_schema)

# # Save results to JSON
# with open("schema_comparison.json", "w") as f:
#     json.dump(comparison_results, f, indent=2)

# print("\n✅ Schema comparison completed! Results saved to 'schema_comparison.json'")


In [ ]:
comparison_results["missing_entities"]

In [151]:
def restructure_properties_single_row(properties_dict):
    """
    Restructures the properties dictionary into a DataFrame where:
    - Each entity is a row.
    - All its missing/extra properties are stored in the same row as separate columns.

    Parameters:
    - properties_dict: Dictionary with entities as keys and list of properties as values.

    Returns:
    - Pandas DataFrame
    """
    reformatted_data = [{"Entity": entity, **{f"Property {i+1}": prop for i, prop in enumerate(props)}}
                        for entity, props in properties_dict.items()]

    return pd.DataFrame(reformatted_data)

In [166]:
import pandas as pd

def comparison_results_to_dataframe(comparison_results):
    """
    Convert comparison results dictionary into multiple Pandas DataFrames.
    
    Returns:
    - missing_entities_df: DataFrame listing missing entities.
    - extra_entities_df: DataFrame listing extra entities.
    - missing_properties_df: DataFrame listing missing properties per entity.
    - extra_properties_df: DataFrame listing extra properties per entity.
    """
    # Convert missing and extra entities to DataFrames
    missing_entities_df = pd.DataFrame(comparison_results.get("missing_entities", []), columns=["Missing Entities"])
    extra_entities_df = pd.DataFrame(comparison_results.get("extra_entities", []), columns=["Extra Entities"])

    # # Convert missing properties to DataFrame
    missing_properties_df = restructure_properties_single_row(comparison_results.get("missing_properties", {}))

    # Convert extra properties to DataFrame
    extra_properties_df = restructure_properties_single_row(comparison_results.get("extra_properties", {}))

    return missing_entities_df, extra_entities_df, missing_properties_df, extra_properties_df


missing_entities_df, extra_entities_df, missing_properties_df, extra_properties_df = comparison_results_to_dataframe(comparison_results)


In [165]:
missing_properties_df[missing_properties_df["Entity"]=="knowledgeasset"]


,Entity,Property 1,Property 2,Property 3,Property 4,Property 5,Property 6,Property 7,Property 8,Property 9,...,Property 21,Property 22,Property 23,Property 24,Property 25,Property 26,Property 27,Property 28,Property 29,Property 30
0,knowledgeasset,creator,part_of,has_part,media,alternate_name,distribution,applies,description,licence,...,location,date_created,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [170]:
missing_entities_df
# extra_entities_df
# missing_properties_df
# # extra_properties_df

,Missing Entities
0,expertgroup
1,purpose
2,cloudcomputeasset
3,protocol
4,jobannouncement
...,...
83,contactdetails
84,audio
85,companyrevenue
86,businesssector
